# 📄 End-to-End RAG Pipeline using LangChain, FAISS, and Llama 3.2

This project demonstrates an end-to-end Retrieval-Augmented Generation (RAG) pipeline using:

- LangChain
- Hugging Face Embeddings
- FAISS Vector Database
- Ollama
- Llama 3.2 (Open Source LLM)

The chatbot answers questions based only on the provided Employee Handbook PDF.

## Step 1: Import Required Libraries

Import the necessary libraries for loading PDF documents.

In [1]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\Hp\AppData\Local\Temp\ipykernel_10388\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\Hp\anaconda3\Anaconda\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2: Load the PDF Document

Load the Employee Handbook into LangChain Document objects.
Each page of the PDF is stored as a separate Document.

In [2]:
loader = PyPDFLoader("../data/Employee Handbook.pdf")
documents = loader.load()

## Step 3: Verify the Loaded Document

Check the number of pages and verify that the PDF has been loaded successfully.

In [3]:
print(type(documents))
print(len(documents))

<class 'list'>
28


## Step 4: Preview the First Page

Display the contents of the first page to verify the extracted text.

In [4]:
print(documents[0])

page_content='©2004 National Council of Nonprofit Associations 
May be duplicated, with attribution, by charitable organizations. 
 
 
 
 
 
 
 
 
 
 
SAMPLE EMPLOYEE HANDBOOK' metadata={'producer': 'Acrobat Distiller 6.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2006-10-16T19:54:33-04:00', 'moddate': '2006-10-16T19:54:33-04:00', 'title': 'Microsoft Word - Sample Employee Handbook for web.doc', 'author': 'JBergin', 'source': '../data/Employee Handbook.pdf', 'total_pages': 28, 'page': 0, 'page_label': '1'}


In [5]:
print(documents[0].page_content)

©2004 National Council of Nonprofit Associations 
May be duplicated, with attribution, by charitable organizations. 
 
 
 
 
 
 
 
 
 
 
SAMPLE EMPLOYEE HANDBOOK


## Step 5: Import the Text Splitter

Import RecursiveCharacterTextSplitter for dividing the document into smaller overlapping chunks.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Step 6: Split the Document into Chunks

Split the document into chunks of 1000 characters with an overlap of 200 characters.

Chunking improves retrieval accuracy while preserving context.

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)

## Step 7: Verify the Chunks

Check the total number of generated chunks and preview one chunk.

In [8]:
print(len(chunks))
#print first chunk
print(chunks[1].page_content)

93
{ORGANIZATION NAME} Employee Handbook 
©2004 National Council of Nonprofit Associations 
May be duplicated, with attribution, by charitable organizations. 
 
ii
EMPLOYEE RECEIPT AND ACCEPTANCE 
I hereby acknowledge receipt of the {Organization Name} Employee Handbook.  I understand 
that it is my continuing responsibility to read and know its contents.  I also understand and agree that 
the Employee Handbook is not an employment contract for any specific period of employment or for 
continuing or long‐term employment.  Therefore, I acknowledge and understand that unless I have a 
written employment agreement with {ORGANIZATION NAME} that provides otherwise, I have the 
right to resign from my employment with {ORGANIZATION NAME} at any time with or without 
notice and with or without cause, and that {ORGANIZATION NAME} has the right to terminate my 
employment at any time with or without notice and with or without cause.


## Step 8: Load the Embedding Model

Load the Hugging Face embedding model that converts text into numerical vectors.

Model used:
- sentence-transformers/all-MiniLM-L6-v2

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 745.84it/s]


## Step 9: Generate a Sample Embedding

Generate an embedding for a sample sentence to understand the output produced by the embedding model.

In [10]:
embedding = embedding_model.embed_query("Hello World")

print(type(embedding))
print(len(embedding))
print(embedding[:10])

<class 'list'>
384
[-0.034477267414331436, 0.031023245304822922, 0.006735002622008324, 0.02610897272825241, -0.03936199098825455, -0.16030247509479523, 0.06692401319742203, -0.006441466510295868, -0.047450482845306396, 0.014758829958736897]


## Step 10: Import FAISS

Import FAISS, a vector database used to store document embeddings for efficient semantic search.

In [11]:
from langchain_community.vectorstores import FAISS

## Step 11: Create the FAISS Vector Store

Generate embeddings for all document chunks and store them in a FAISS vector database.

In [12]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [13]:
vectorstore.save_local("faiss_index")

This stores your FAISS database on disk.

After this, you don't need to run Cell 1 again unless:

you add new documents
you change chunk size
you change embedding model

## Step 12: Perform Semantic Search

Retrieve the most relevant document chunks using semantic similarity instead of keyword matching.

In [15]:
results = vectorstore.similarity_search(
    "How many paid leaves do employees get?",
    k=2
)

## Step 13: Display Retrieved Chunks

Display the retrieved chunks to verify that FAISS returns relevant information.

In [16]:
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print(doc.page_content)
    print("-" * 50)
    print(doc.metadata)

Result 1
employees the difference between his or her salary and any amount paid by 
the government, unless prohibited by law, up to a maximum of ten days.  If an 
employee is required to serve more than ten days of jury duty, 
{ORGANIZATION NAME} will provide the employee with unpaid leave. 
Employees must provide {ORGANIZATION NAME} a copy of proof of
--------------------------------------------------
{'producer': 'Acrobat Distiller 6.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2006-10-16T19:54:33-04:00', 'moddate': '2006-10-16T19:54:33-04:00', 'title': 'Microsoft Word - Sample Employee Handbook for web.doc', 'author': 'JBergin', 'source': '../data/Employee Handbook.pdf', 'total_pages': 28, 'page': 15, 'page_label': '16'}
Result 2
10 
 
annually by the {ORGANIZATION NAME} Board of Directors.  Information about 
{ORGANIZATION NAME}ʹs retirement plan will be provided to the employee at the time of 
employment. 
 
E.  Tax Deferred Annuity Plan 
  {ORGANIZATION

## Step 14: Load the Open Source LLM

Load the Llama 3.2 model using Ollama for local inference.

In [17]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

## Step 15: Create a Retriever

Convert the FAISS vector store into a retriever that returns the top three most relevant document chunks.

In [18]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

## Step 16: Import Chat Prompt Template

Import LangChain's ChatPromptTemplate to construct prompts dynamically.

In [20]:
from langchain_core.prompts import ChatPromptTemplate

## Step 17: Create the Prompt Template

Create a prompt that instructs the language model to answer questions only from the retrieved context.

In [21]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful HR policy assistant.

Answer the question ONLY using the context provided.

Rules:
- Give a concise answer (2-5 sentences).
- Include important conditions or exceptions if they directly affect the answer.
- Do not copy the entire context.
- Do not add information outside the context.
- Preserve exact numbers, time periods, and eligibility conditions. Do not generalize beyond what the context states.
- Treat each year of employment separately. Never combine or assume the same allowance applies across multiple years. Report each year's allowance exactly as stated in the context.
- If the answer is not in the context, say:
"I don't know."

Context:
{context}

Question:
{question}

Answer:
""")

## Step 18: Execute the End-to-End RAG Pipeline

Execute the complete Retrieval-Augmented Generation (RAG) workflow by retrieving relevant document chunks, constructing the prompt, and generating the final response using the Llama 3.2 language model.

In [24]:
question = "According to the context, how many vacation days does a full-time employee earn during the second year of employment? Answer with the number of days only."

print("1")
docs = retriever.invoke(question)

print("2")
print("Number of documents retrieved:", len(docs))

for i, doc in enumerate(docs):
    print("\n--- Document", i+1, "---")
    print(doc.page_content[:500])

print("3")
context = "\n\n".join(doc.page_content for doc in docs)

print("4")
final_prompt = prompt.format(
    context=context,
    question=question
)
print(final_prompt)

print("5")
print("\n--- CONTEXT SENT TO MODEL ---")
print(context)
response = llm.invoke(final_prompt)

print(response.content)

1
2
Number of documents retrieved: 3

--- Document 1 ---
11 
 
  During the first 90 days of employment full‐ and part‐time employees will not earn 
Vacation benefits.  During the remaining nine months of first year employment, a full‐time 
employee will earn two weeks (10 days) of paid Vacation. 
 
  Full‐time employees will continue to earn two weeks (10 days) of Vacation during the 
second year of employment.  In their third year of employment, full‐time employees will earn 
three weeks (15 days) of Vacation.  During the fourth year and thereafter,

--- Document 2 ---
increments, using the appropriate leave request form..   
 
  Employees are expected to use Vacation benefits in the fiscal year in which Vacation is 
earned.  Employees may carry over unused Vacation from one year to the next only with the 
approval of the Executive Director.  Employees may not accrue more than the maximum leave 
they are allowed.  Once an employee reaches his or her annual ceiling, the employee cease

In [ ]:
question = "How many paid vacation days does a full-time employee get in each of the first four years?"

docs = retriever.invoke(question)
context = "\n\n".join(doc.page_content for doc in docs)

final_prompt = prompt.format(
    context=context,
    question=question
)

response = llm.invoke(final_prompt)
print(response.content)

A full-time employee will earn 2 weeks (10 days) of paid vacation in their:

1st year: 10 days
2nd year: 10 days
3rd year: 15 days
4th year and thereafter: 20 days


In [29]:
question = "What is the company's policy on paid paternity leave for twins?"

docs = retriever.invoke(question)
context = "\n\n".join(doc.page_content for doc in docs)

final_prompt = prompt.format(
    context=context,
    question=question
)

response = llm.invoke(final_prompt)
print(response.content)

I don't know. The provided context does not mention a specific policy for paid paternity leave, including for twins. It only mentions bereavement leave and parental leave policies in general.


In [30]:
question = "How many sick leave days do employees earn per month?"

docs = retriever.invoke(question)
context = "\n\n".join(doc.page_content for doc in docs)

final_prompt = prompt.format(
    context=context,
    question=question
)

response = llm.invoke(final_prompt)
print(response.content)

Full-time employees earn 1 day (7 hours) of sick leave benefits per month. Part-time employees receive prorated sick leave benefits, with a rate of 3.5 hours per month if they work 17.5 hours per week.


In [42]:
test_questions = [
    {
        "question": "How many sick leave hours do full-time employees earn per month?",
        "expected": "7 hours"
    },
    {
        "question": "What is the maximum sick leave full-time employees can accumulate?",
        "expected": "30 days"
    },
    {
        "question": "How many paid vacation days does a full-time employee get in the third year?",
        "expected": "15 days"
    },
    {
        "question": "What is the company's policy on paid paternity leave for twins?",
        "expected": "I don't know"
    },
    {
        "question": "During the first 90 days of employment, do full-time employees earn paid vacation?",
        "expected": "will not earn any paid vacation"
    },
    {
        "question": "How many paid vacation days does a full-time employee get in the fourth year?",
        "expected": "20 days"
    },
    {
        "question": "Are temporary employees eligible for paid sick leave?",
        "expected": "not eligible"
    }

]

print("Total evaluation questions:", len(test_questions))

Total evaluation questions: 7


In [43]:

passed = 0

for i, test in enumerate(test_questions, start=1):
    question = test["question"]

    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    final_prompt = prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(final_prompt)
    answer = response.content

    expected = test["expected"]

    if expected.lower() in answer.lower():
        status = "PASS"
        passed += 1
    else:
        status = "CHECK"

    print(f"\n--- Test {i}: {status} ---")
    print("Question:", question)
    print("Expected:", expected)
    print("Actual:", answer)

print(f"\nResults: {passed}/{len(test_questions)} passed")



--- Test 1: PASS ---
Question: How many sick leave hours do full-time employees earn per month?
Expected: 7 hours
Actual: Full-time employees earn 7 hours of sick leave per month.

--- Test 2: PASS ---
Question: What is the maximum sick leave full-time employees can accumulate?
Expected: 30 days
Actual: Full-time employees are allowed to accumulate a maximum of 30 days (210 hours) of unused sick leave benefits.

--- Test 3: PASS ---
Question: How many paid vacation days does a full-time employee get in the third year?
Expected: 15 days
Actual: In the third year of employment, a full-time employee will earn 3 weeks (15 days) of paid vacation.

--- Test 4: PASS ---
Question: What is the company's policy on paid paternity leave for twins?
Expected: I don't know
Actual: I don't know. The provided context does not mention a specific policy for paid paternity leave, including for twins. It only mentions bereavement leave and parental leave policies in general.

--- Test 5: PASS ---
Question